In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6" 

In [2]:
from pdf_processor import PDFProcessor
from section_parser import SectionHierarchicalParser
from section_merger import SectionMerger
from base_processor import BaseProcessor
from translate_infer import *
from tqdm import tqdm
import re

import re

def num_cut(s,target):
    j = 0
    count = 0

    s = s.split(' ')
    target = target.split(' ')

    for i in range(len(s)):
        if s[i] == target[j]:
            j += 1
            if j == len(target):
                j = 0
                count += 1
        else:
            j = 0
            if (s[i] == target[j]):
                j += 1

    return count

def LMAS(s):
    words = s.split()
    dict_count = {}
    for dist in range(1, len(words) - 1):
        for i in range(0,len(words) - dist + 1):
            j = i + dist
            if dist == 1:
                dict_count[words[i]] = dict_count.get(words[i],0) + 1
            else:
                substr = " ".join(words[i:j])
                dict_count[substr] = dict_count.get(substr,0) + 1

    count = -1
    save_str = ""
    
    for k,v in dict_count.items():
        str_temp = k
        if len(str_temp) >= len(s) * 0.5:
            continue
        count_temp = num_cut(s, str_temp)
        if count_temp == 1: continue

        if count_temp * (len(str_temp) + 1) > count * (len(save_str) + 1):
            count = count_temp
            save_str = str_temp
            # print(save_str," ",count," ",v)
            
    return save_str,count

def replace_LMAS(s):
    s = s.strip()
    s = re.sub(r"\s+"," ",s)
    
    pattern,count = LMAS(s)
    len_word = len(pattern.split(' '))
    if count == -1: return s
    
    words = s.split(' ')
    word = pattern
    if len_word == 1:
        if count <= 5: return s
        if count <= 0.05 * len(s):
            return s
        if word == words[-1]:
            save = -1
            for j in range(len(words)):
                if word == words[j]:
                    continue
                else:
                    save = j
                    break

            s = s[:j]
            return s
            
    if len_word <= 3 and count <= 5:
        return s
    
    idx = s.find(pattern)
    s = s[:idx + len(pattern)]

    return s

def add_newline_before_headings(text):
    # Biểu thức regex nhận diện các tiêu đề dạng số hoặc chữ cái, ví dụ: "1.", "2.", "III.", "Part I"
    pattern = r'(\b\d+\.\s|\b[A-Z]+\.\s|Part\s+[A-Z]+|\b[a-z]\)\s|\b[a-z]\/\s)'
    
    # Thêm newline trước các đề mục tìm thấy
    new_text = re.sub(pattern, r'\n\1', text)
    
    return new_text.strip()  # Xóa khoảng trắng không cần thiết ở đầu và cuối

def get_depth(item):
    if isinstance(item, str) or (isinstance(item, dict) and len(item['child']) == 0):
        return 1
    return 1 + max(get_depth(child) for child in item['child'])

class TranslatePipeline(BaseProcessor):
    RETRY = False
    
    def __init__(self):
        super().__init__()

        self.pdf = PDFProcessor()
        self.section_parser = SectionHierarchicalParser()
        self.section_merger = SectionMerger()
        self.model = InferEngine()
        
        self.batch_size = 4
        self.min_length = 300
        self.max_length = 300
    
    def raw_process(self, docs, en_vi=False):
        result = self.model.inference(docs, en_vi) 

        return result
    
    def process(self, path=None, doc=None, en_vi = False, debug=False):
        if path is not None:
            item = path
            item = self.pdf.process(item)
            #not support table
            
            item = item['sections']
        elif doc is not None:
            item = doc.split('\n')

        item = self.section_parser.section_parsing(item, max_level=self.max_length)
        item = {'value': '', 'child': item}
        current_target_level = min(get_depth(item) - 1, self.max_length)
        while current_target_level >= 0:
            item = self.section_merger.merge(item, current_target_level, max_length=self.max_length)
            current_target_level -= 1            
        
        text_value = [doc['value'] for doc in item['child']]
        batchs = []
        outputs = []
        
        batch = []
        for txt in text_value:
            batch.append(txt)
            if len(batch) == self.batch_size:
                batchs.append(batch)
                batch = []

        for batch in tqdm(batchs):
            result = self.model.inference(batch, en_vi)
            for res in result:
                # res = res.replace(' - ',' \n- ')
                # res = add_newline_before_headings(res).replace('en:','').replace('vi:','')
                res = res.replace('en:','').replace('vi:','')
                res = replace_LMAS(res)
                outputs.append(res)
        
        docs = '\n'.join(outputs)
        
        return docs,outputs

        # return batchs

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
trans = TranslatePipeline()

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
output,list_output = trans.process(path = "/raid/phundh/translateFeature/pdf/06_2024_TT-BKHDT_608043.pdf")

  0%|          | 0/13 [00:00<?, ?it/s]/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
100%|██████████| 13/13 [01:34<00:00,  7.26s/it]


In [4]:
batch_output = trans.process(path = "/raid/phundh/translateFeature/pdf/06_2024_TT-BKHDT_608043.pdf")

In [7]:
print(output.replace('\\n','\n'))

``` <SECTION> 
 
Chapter IGENERAL PROVISIONSArticle1. Scope of RegulationThis Circular guides the provision and publication of information on contractor selection and bidding dossier templates, including: 1. Form for formulation, appraisal, and approval of the overall contractor selection plan, contractor selection plan for projects, and procurement estimates as prescribed in Articles1 and2 of the Bidding Law; 2. Bidding dossier templates on the National Bidding Network System for packages providing consulting services, non-consulting services, goods (excluding medicines), and construction within the scope of the Bidding Law, which are organized through open bidding, limited bidding, and competitive domestic bidding in the one-stage one-envelope manner, and one-stage two-envelope method. Article2. Applicable Entities1. Organizations and individuals related to contractor selection within the scope of regulation specified in Article1 of this
'Chapter IGENERAL PROVISIONSArticle3. Definiti

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import time
import torch._dynamo as torchdynamo
import torch
from kernl.model_optimization import optimize_model

In [54]:
torchdynamo.config.suppress_errors = True

In [33]:
torchdynamo.config.cache_size_limit

8

In [6]:
# default cache size needs to be increased to store the many graphs with generative models
torchdynamo.config.cache_size_limit = 512

model_name = '/raid/phundh/trans_model/checkpoint-33942'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [61]:
# default cache size needs to be increased to store the many graphs with generative models
#torchdynamo.config.cache_size_limit = 512

model_name = 'VietAI/envit5-translation'
model1 = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model1 = model.eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [48]:
input_ids = tokenizer(batch_output[0], return_tensors="pt", padding=True, max_length = 1024).input_ids.to('cuda')

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [8]:
a = time.time()

input_ids = tokenizer(batch_output[0] + batch_output[1] + batch_output[2], return_tensors="pt", padding=True, max_length = 1024).input_ids.to('cuda')
outputs = model.generate(input_ids,max_length=1024)
list_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)

b = time.time()

print(b - a)

14.087865114212036


In [50]:
print(list_output)

['en: \'Article1. Scope of RegulationThis Circular guides the provision and posting of information on contractor selection and bidding dossier templates, including: 1. Form for formulation, appraisal, and approval of the overall contractor selection plan, contractor selection plan for projects, and procurement estimates in accordance with Articles1 and2 of the Bidding Law; 2. Bidding dossier templates on the National Bidding Network System for packages providing consulting services, non-consulting services, goods (excluding medicines), and construction within the scope of the Bidding Law, which are organized through open bidding, limited bidding, and competitive domestic procurement through one-stage one-envelope bidding, one-stage two-envelope bidding.\' \'Article2. Applicable Entities1. Organizations and individuals related to contractor selection within the scope of regulation specified in Article1 of this Circular.2. Organizations and individuals with contractor selection activitie

In [51]:
with torch.inference_mode(), torch.autocast(dtype=torch.float16, cache_enabled=True, device_type="cuda"):
    torch.cuda.synchronize()
    start = time.perf_counter()
    outputs = model.generate(input_ids,max_length=1024)
    torch.cuda.synchronize()
    latency_baseline = time.perf_counter() - start
    print(latency_baseline)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

5.377101589925587
['en: \'Article1. Scope of RegulationThis Circular guides the provision and posting of information on contractor selection and bidding dossier templates, including: 1. Form for formulation, appraisal, and approval of the overall contractor selection plan, contractor selection plan for projects, and procurement estimates in accordance with Articles1 and2 of the Bidding Law; 2. Bidding dossier templates on the National Bidding Network System for packages providing consulting services, non-consulting services, goods (excluding medicines), and construction within the scope of the Bidding Law, which are organized through open bidding, limited bidding, and competitive domestic procurement through one-stage one-envelope bidding, one-stage two-envelope bidding.\' \'Article2. Applicable Entities1. Organizations and individuals related to contractor selection within the scope of regulation specified in Article1 of this Circular.2. Organizations and individuals with contractor s

In [52]:
optimize_model(model.encoder)
optimize_model(model.decoder)

In [55]:
with torch.inference_mode(), torch.autocast(dtype=torch.float16, cache_enabled=True, device_type="cuda"):
    torch.cuda.synchronize()
    start = time.perf_counter()
    outputs = model.generate(input_ids,max_length=1024)
    torch.cuda.synchronize()
    latency_baseline = time.perf_counter() - start
    print(latency_baseline)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

118.33909138781019
['en', 'vi', 'vi', 'vi']


In [57]:
with torch.inference_mode(), torch.autocast(dtype=torch.float16, cache_enabled=True, device_type="cuda"):
    torch.cuda.synchronize()
    start = time.perf_counter()
    input_ids = tokenizer(batch_output[0] + batch_output[1], return_tensors="pt", padding=True, max_length = 1024).input_ids.to('cuda')
    outputs = model.generate(input_ids,max_length=1024)
    torch.cuda.synchronize()
    latency_baseline = time.perf_counter() - start
    print(latency_baseline)
    print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:2692: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
W1015 03:03:17.493124 140156614231872 torch/_dynamo/convert_frame.py:824] WON'T CONVERT forward /home/minhnh/python_venv/nlp/lib/python3.9/site-packages/transformers/models/t5/modeling_t5.py line 242 
W1015 03:03:17.493124 140156614231872 torch/_dynamo/convert_frame.py:824] due to: 
W1015 03:03:17.493124 140156614231872 torch/_dynamo/convert_frame.py:824] Traceback (most recent call last):
W1015 03:03:17.493124 140156614231872 torch/_dynamo/convert_frame.py:824]   File "/home/minhnh/python_venv/nlp/lib/python3.9/site-packages/torch/_dynamo/convert_frame.py", line 786, in _convert_frame
W1015 03:03:17.493124 140156614231872 torch/_dynamo/convert_frame.py:824]     result = inner_convert(
W1015 03:03:17.493124 140156614231872 t

5.773338630795479
['en: \'Article1. Scope of RegulationThis Circular guides the provision and posting of information on contractor selection and bidding dossier templates, including: 1. Form for formulation, appraisal, and approval of the overall contractor selection plan, contractor selection plan for projects, and procurement estimates in accordance with Articles1 and2 of the Bidding Law; 2. Bidding dossier templates on the National Bidding Network System for packages providing consulting services, non-consulting services, goods (excluding medicines), and construction within the scope of the Bidding Law, which are organized through open bidding, limited bidding, and competitive domestic procurement through one-stage one-envelope bidding, one-stage two-envelope bidding.\' \'Article2. Applicable Entities1. Organizations and individuals related to contractor selection within the scope of regulation specified in Article1 of this Circular.2. Organizations and individuals with contractor s